# Physics-Informed Material Ranking Framework
## Complete Demonstration: CHGNet + ML + Physics Descriptors → Composite Score

**Objective:** Rank lithium-ion conducting materials using:
- CHGNet structure relaxation & energy prediction
- Physics-based descriptors (tEF, tFI)
- Non-linear state-based framework (T, S, F)
- Synthesizability feasibility scoring
- Final composite ranking (0–100)

**Dataset:** Obliex crystal structures + experimental data

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# SECTION 1: LOAD & PARSE CRYSTAL STRUCTURES (CIF FILES)
# ══════════════════════════════════════════════════════════════════════════════

import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from typing import Dict, List, Optional, Tuple

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')

# Import our framework modules
import sys
sys.path.insert(0, r'c:\Users\saisu\OneDrive\Desktop\Final_ validation')

from material_ranking_framework import *
from data_pipeline_integration import *
from evaluation_pipeline import *

print("✅ Imports successful")

# Configuration
CIF_DIR = Path(r"c:\Users\saisu\OneDrive\Desktop\Final_ validation\Obliex_cifs\all_randomized_cifs")
CSV_PATH = Path(r"c:\Users\saisu\OneDrive\Desktop\Final_ validation\Obliex.csv")
OUTPUT_DIR = Path(r"c:\Users\saisu\OneDrive\Desktop\Final_ validation\ranking_results")
OUTPUT_DIR.mkdir(exist_ok=True)

print(f"CIF directory: {CIF_DIR}")
print(f"CSV path: {CSV_PATH}")
print(f"Output directory: {OUTPUT_DIR}")

# Check CIF availability
cif_files = list(CIF_DIR.glob("*.cif"))
print(f"\n📁 Found {len(cif_files)} CIF files")
if cif_files:
    print(f"   Sample: {cif_files[:3]}")

## Section 1: Load and Parse Crystal Structures

Extract lattice parameters from CIF files and compute initial unit-cell volume:

$$V = abc\sqrt{1 - \cos^2\alpha - \cos^2\beta - \cos^2\gamma + 2\cos\alpha\cos\beta\cos\gamma}$$

In [ ]:
try:
    from pymatgen.core import Structure
    from pymatgen.symmetry.analyzer import SpacegroupAnalyzer
    
    # Load a few example CIF files
    test_cifs = sorted(list(CIF_DIR.glob("*.cif")))[:5]  # First 5 for demo
    
    structures_demo = {}
    for cif_path in test_cifs:
        try:
            struct = Structure.from_file(str(cif_path))
            structures_demo[cif_path.stem] = struct
        except Exception as e:
            print(f"⚠️  Failed to load {cif_path.stem}: {e}")
    
    print(f"✅ Loaded {len(structures_demo)} example structures")
    
    # Extract lattice parameters and compute volumes
    lattice_data = []
    for material_id, struct in structures_demo.items():
        a, b, c = struct.lattice.abc
        alpha, beta, gamma = struct.lattice.angles
        
        # Compute volume using general formula
        cos_alpha, cos_beta, cos_gamma = np.cos(np.radians([alpha, beta, gamma]))
        volume = a * b * c * np.sqrt(
            1 - cos_alpha**2 - cos_beta**2 - cos_gamma**2 + 2*cos_alpha*cos_beta*cos_gamma
        )
        
        lattice_data.append({
            'material_id': material_id,
            'a': a, 'b': b, 'c': c,
            'alpha': alpha, 'beta': beta, 'gamma': gamma,
            'volume_initial': volume,
            'n_atoms': len(struct),
            'density': struct.density,
        })
    
    df_lattice = pd.DataFrame(lattice_data)
    print("\n📊 Lattice Parameters (First 5 Materials):")
    print(df_lattice.to_string(index=False))
    
except ImportError:
    print("⚠️  PyMatGen not available - install with: pip install pymatgen")

## Section 2-5: Physics-Based Descriptors & State Framework

The framework computes:

**Transport Efficiency Factor (tEF):**
$$tEF = \frac{\sigma}{E_a}$$

**Transport Frustration Index (tFI):** proxy from disorder metrics

**Three Interacting States:**
- Transport State: $T = f(\sigma, E_a, tEF)$
- Stability State: $S = f(E_{hull}, \Delta V, K_B)$
- Frustration State: $F = f(tFI, disorder)$

**Non-Linear Interaction Model:**
$$Score_{raw} = T^\alpha \times S^\beta \times e^{-\gamma F}$$

In [ ]:
# Example: Create synthetic materials with realistic properties for demo
demo_materials = [
    MaterialProperties(
        material_id="Material_A",
        composition="Li3PS4",
        structural=StructuralMetrics(volume_initial=100.0, density=1.8),
        thermodynamic=ThermodynamicProperties(
            formation_energy=-5.2,
            energy_above_hull=0.05,
            magnetic_moment=0.1,
        ),
        transport=TransportProperties(
            ionic_conductivity=0.001,
            migration_barrier=0.35,
        ),
        mechanical=MechanicalProperties(bulk_modulus=45, shear_modulus=28),
        disorder_metric=0.3,
    ),
    MaterialProperties(
        material_id="Material_B",
        composition="Li3PO4",
        structural=StructuralMetrics(volume_initial=95.0, density=1.9),
        thermodynamic=ThermodynamicProperties(
            formation_energy=-6.1,
            energy_above_hull=0.02,
            magnetic_moment=0.05,
        ),
        transport=TransportProperties(
            ionic_conductivity=0.0005,
            migration_barrier=0.45,
        ),
        mechanical=MechanicalProperties(bulk_modulus=50, shear_modulus=32),
        disorder_metric=0.2,
    ),
    MaterialProperties(
        material_id="Material_C",
        composition="Li2SO4",
        structural=StructuralMetrics(volume_initial=110.0, density=1.7),
        thermodynamic=ThermodynamicProperties(
            formation_energy=-4.8,
            energy_above_hull=0.15,
            magnetic_moment=0.2,
        ),
        transport=TransportProperties(
            ionic_conductivity=0.0008,
            migration_barrier=0.55,
        ),
        mechanical=MechanicalProperties(bulk_modulus=40, shear_modulus=25),
        disorder_metric=0.5,
    ),
]

print("✅ Created 3 example materials for demonstration")

# Compute descriptors for each material
print("\n📊 PHYSICS DESCRIPTORS:")
print("=" * 100)

descriptor_results = []
for mat in demo_materials:
    tef = PhysicsDescriptors.transport_efficiency_factor(
        mat.transport.ionic_conductivity,
        mat.transport.migration_barrier,
    )
    
    tfi = PhysicsDescriptors.transport_frustration_index(
        mat.disorder_metric,
    )
    
    disorder = PhysicsDescriptors.structural_disorder_proxy(
        mat.structural.volume_change,
        mat.thermodynamic.magnetic_moment,
        space_group_number=225,  # Cubic example
    )
    
    descriptor_results.append({
        'material_id': mat.material_id,
        'tEF': tef,
        'tFI': tfi,
        'disorder': disorder,
        'sigma': mat.transport.ionic_conductivity,
        'E_a': mat.transport.migration_barrier,
        'E_hull': mat.thermodynamic.energy_above_hull,
    })

df_descriptors = pd.DataFrame(descriptor_results)
print(df_descriptors.to_string(index=False))

print("\n✅ Physics descriptors computed")

## Section 6: Build State-Based Interaction Framework

Compute the three physical states and their non-linear interaction.

In [ ]:
# Compute states for each material
print("📊 PHYSICAL STATES:")
print("=" * 100)

state_results = []
for mat in demo_materials:
    tef = PhysicsDescriptors.transport_efficiency_factor(
        mat.transport.ionic_conductivity,
        mat.transport.migration_barrier,
    )
    
    tfi = PhysicsDescriptors.transport_frustration_index(mat.disorder_metric)
    
    # Build states
    T = StateBuilder.transport_state(
        mat.transport.ionic_conductivity,
        mat.transport.migration_barrier,
        tef,
    )
    
    S = StateBuilder.stability_state(
        mat.thermodynamic.energy_above_hull,
        mat.structural.volume_change,
        mat.mechanical.bulk_modulus if mat.mechanical else 50,
    )
    
    F = StateBuilder.frustration_state(tfi, mat.disorder_metric)
    
    states = MaterialStates(T=T, S=S, F=F)
    
    # Compute feasibility
    feasibility = SynthesizabilityScore.feasibility_combined(
        mat.thermodynamic.energy_above_hull,
        mat.structural.volume_change,
    )
    
    state_results.append({
        'material_id': mat.material_id,
        'Transport (T)': T,
        'Stability (S)': S,
        'Frustration (F)': F,
        'Feasibility': feasibility,
    })

df_states = pd.DataFrame(state_results)
print(df_states.to_string(index=False))

print("\n✅ Physical states computed")

## Section 9-10: Composite Material Score & Ranking

Apply non-linear interaction model with different parameter sets to demonstrate sensitivity.

In [ ]:
# Standard ranking with default parameters
print("\n🏆 COMPOSITE MATERIAL RANKING (0-100 SCORE)")
print("=" * 100)

# Use default interaction parameters
params_default = InteractionParameters(alpha=2.0, beta=1.5, gamma=3.0)
ranker = MaterialRanker(params_default)

# Rank the demo materials
results_df = ranker.rank_batch(demo_materials)

# Display results
display_cols = ['rank', 'material_id', 'composition', 'transport_state', 
                'stability_state', 'frustration_state', 'feasibility', 'score_normalized']
print("\nRanking Results:")
print(results_df[display_cols].to_string(index=False))

# Print statistics
stats = ranker.summary_statistics()
print(f"\n📊 RANKING STATISTICS:")
print(f"   Materials evaluated: {stats['n_materials']}")
print(f"   Score mean: {stats['score_mean']:.1f} ± {stats['score_std']:.1f}")
print(f"   Score range: [{stats['score_min']:.1f}, {stats['score_max']:.1f}]")
print(f"   Average feasibility: {stats['feasibility_mean']:.3f}")

print("\n✅ Ranking complete")

## Sensitivity Analysis: Parameter Effects on Ranking

Demonstrate how tuning $\alpha$, $\beta$, and $\gamma$ affects material rankings.

In [ ]:
# Compare different interaction parameter configurations
print("\n🔬 SENSITIVITY ANALYSIS: Parameter Effect on Rankings")
print("=" * 100)

param_configs = {
    'Default (α=2.0, β=1.5)': InteractionParameters(alpha=2.0, beta=1.5, gamma=3.0),
    'Transport-heavy (α=3.0, β=1.0)': InteractionParameters(alpha=3.0, beta=1.0, gamma=2.0),
    'Stability-heavy (α=1.5, β=2.5)': InteractionParameters(alpha=1.5, beta=2.5, gamma=2.0),
    'Balanced (α=1.5, β=1.5)': InteractionParameters(alpha=1.5, beta=1.5, gamma=2.5),
}

comparison_results = {}
for config_name, params in param_configs.items():
    ranker_temp = MaterialRanker(params)
    df_temp = ranker_temp.rank_batch(demo_materials)
    comparison_results[config_name] = df_temp.set_index('material_id')['score_normalized'].to_dict()

df_comparison = pd.DataFrame(comparison_results).round(1)
print("\nComposite Scores under Different Parameter Sets:")
print(df_comparison)

# Visualize sensitivity
fig, ax = plt.subplots(figsize=(10, 5))
df_comparison.T.plot(kind='bar', ax=ax)
ax.set_xlabel('Configuration')
ax.set_ylabel('Composite Score (0-100)')
ax.set_title('Ranking Sensitivity to Interaction Parameters')
ax.legend(title='Material', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

print("\n✅ Sensitivity analysis complete")

## Visualizations & Physical Insights

Analyze state distributions and feature contributions to ranking.

In [ ]:
# Visualize state distributions
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# State values
materials_list = results_df['material_id'].tolist()
T_vals = results_df['transport_state'].tolist()
S_vals = results_df['stability_state'].tolist()
F_vals = results_df['frustration_state'].tolist()
scores = results_df['score_normalized'].tolist()

# Plot 1: Transport vs Stability
axes[0, 0].scatter(T_vals, S_vals, s=200, c=scores, cmap='viridis', edgecolors='black', linewidth=2)
axes[0, 0].set_xlabel('Transport State (T)', fontsize=11)
axes[0, 0].set_ylabel('Stability State (S)', fontsize=11)
axes[0, 0].set_title('Transport vs Stability (colored by score)', fontsize=12, fontweight='bold')
for i, txt in enumerate(materials_list):
    axes[0, 0].annotate(txt, (T_vals[i], S_vals[i]), fontsize=9)

# Plot 2: Stability vs Frustration
axes[0, 1].scatter(S_vals, F_vals, s=200, c=scores, cmap='viridis', edgecolors='black', linewidth=2)
axes[0, 1].set_xlabel('Stability State (S)', fontsize=11)
axes[0, 1].set_ylabel('Frustration State (F)', fontsize=11)
axes[0, 1].set_title('Stability vs Frustration', fontsize=12, fontweight='bold')
for i, txt in enumerate(materials_list):
    axes[0, 1].annotate(txt, (S_vals[i], F_vals[i]), fontsize=9)

# Plot 3: State heatmap
state_matrix = np.array([T_vals, S_vals, F_vals])
im = axes[1, 0].imshow(state_matrix, cmap='RdYlGn', aspect='auto', vmin=0, vmax=1)
axes[1, 0].set_yticks([0, 1, 2])
axes[1, 0].set_yticklabels(['T (Transport)', 'S (Stability)', 'F (Frustration)'])
axes[1, 0].set_xticks(range(len(materials_list)))
axes[1, 0].set_xticklabels(materials_list)
axes[1, 0].set_title('Physical States Heatmap', fontsize=12, fontweight='bold')
plt.colorbar(im, ax=axes[1, 0])

# Plot 4: Ranking
axes[1, 1].barh(materials_list, scores, color='steelblue', edgecolor='black', linewidth=2)
axes[1, 1].set_xlabel('Composite Score (0-100)', fontsize=11)
axes[1, 1].set_title('Material Ranking', fontsize=12, fontweight='bold')
axes[1, 1].set_xlim(0, 100)

plt.tight_layout()
plt.show()

print("✅ Visualizations complete")

## Section 11: Full Batch Evaluation with Obliex Dataset

Run the complete end-to-end pipeline on the actual Obliex crystal structures.

In [ ]:
print("\n" + "="*100)
print("FULL BATCH EVALUATION: OBLIEX DATASET")
print("="*100)

# Initialize evaluation pipeline
params = InteractionParameters(
    alpha=2.0,     # Transport exponent
    beta=1.5,      # Stability exponent
    gamma=3.0,     # Frustration penalty
)

pipeline = MaterialEvaluationPipeline(
    model_dir=None,  # No pretrained model for demo
    api_key=None,    # No MP API for demo
    output_dir=str(OUTPUT_DIR),
    interaction_params=params,
)

# Run evaluation on first N CIF files
N_MATERIALS = 10  # Demo: evaluate 10 materials (increase for full run)
print(f"\n📂 Evaluating first {N_MATERIALS} CIF files from {CIF_DIR}...")

try:
    results_batch, stats = pipeline.evaluate_batch(
        str(CIF_DIR),
        pattern="*.cif",
        limit=N_MATERIALS,
        verbose=True,
    )
    
    print(f"\n✅ Batch evaluation complete!")
    print(f"\nTop 5 ranked materials:")
    print(results_batch[['rank', 'material_id', 'composition', 'score_normalized']].head(5).to_string(index=False))
    
except Exception as e:
    print(f"⚠️  Batch evaluation error: {e}")
    print("   (This is expected if CIF files are not in expected format)")
    print("\nUsing demo materials for remainder of analysis...")

## Summary: Physics-Informed Material Ranking Framework

### Key Components Implemented:

1. **Physics-Based Descriptors**
   - Transport Efficiency Factor (tEF) = σ / E_a
   - Transport Frustration Index (tFI) from structural disorder
   - Disorder proxy from symmetry, volume change, magnetic moments

2. **State-Based Framework**
   - Transport State: T = f(σ, E_a, tEF)
   - Stability State: S = f(E_hull, ΔV, K_B)
   - Frustration State: F = f(tFI, disorder)

3. **Non-Linear Interaction Model**
   - Score_raw = T^α × S^β × e^(-γF)
   - Tunable parameters α, β, γ enable sensitivity analysis
   - Captures physics interactions missing in linear scoring

4. **Synthesizability Assessment**
   - Feasibility = exp(-E_hull / kT) × exp(-|ΔV|)
   - Thermodynamic and structural stability proxy
   - Hard constraints for unrealistic materials

5. **Final Ranking (0–100)**
   - Normalized composite scores
   - Rank ordering by multi-objective optimization
   - Interpretable decision output

### Patent-Relevant Innovation:
> A system and method for evaluating materials by computing a composite score derived from machine learning predictions of transport properties, physics-based descriptors (tEF, tFI), thermodynamic and structural stability measures, and a non-linear interaction framework combining multiple physical states to generate a bounded ranking score.

### Usage Patterns:
```python
# Quick evaluation
results_df, pipeline = run_full_evaluation(
    "path/to/cifs/",
    model_dir="path/to/models/",
    alpha=2.0, beta=1.5, gamma=3.0
)

# Filter candidates
promising = pipeline.filter_by_criteria(
    score_min=70,
    feasibility_min=0.5,
    frustration_state_max=0.4
)

# Sensitivity analysis
compare_interaction_parameters(materials, {
    'config_1': InteractionParameters(...),
    'config_2': InteractionParameters(...),
})
```

In [ ]:
# QUICK REFERENCE: Framework API
print("""
╔════════════════════════════════════════════════════════════════════════════════╗
║                    FRAMEWORK QUICK REFERENCE                                  ║
╚════════════════════════════════════════════════════════════════════════════════╝

MODULE 1: material_ranking_framework.py
─────────────────────────────────────────
Core physics implementation:
  • PhysicsDescriptors.transport_efficiency_factor(σ, E_a) → tEF
  • PhysicsDescriptors.transport_frustration_index(disorder) → tFI
  • StateBuilder.transport_state(σ, E_a, tef) → T ∈ [0,1]
  • StateBuilder.stability_state(E_hull, ΔV, K) → S ∈ [0,1]
  • StateBuilder.frustration_state(tfi, disorder) → F ∈ [0,1]
  • InteractionModel.composite_score_raw(T, S, F, α, β, γ) → score
  • MaterialRanker.rank_batch(materials) → DataFrame (0-100 scores)

MODULE 2: data_pipeline_integration.py
────────────────────────────────────────
Data pipeline:
  • CIFProcessor.load_structure(cif_path) → Structure
  • CIFProcessor.extract_basic_metrics(structure) → StructuralMetrics
  • CHGNetPredictor.predict_structure(structure, relax=True) → {E, F, V}
  • SurrogateModelWrapper.predict_barrier(features) → E_a
  • MaterialDataPipeline.process_cif(cif_path) → MaterialProperties
  • MaterialDataPipeline.process_batch(cif_dir) → [MaterialProperties]

MODULE 3: evaluation_pipeline.py
──────────────────────────────────
End-to-end workflow:
  • MaterialEvaluationPipeline.evaluate_batch(cif_dir) → (DataFrame, stats)
  • MaterialEvaluationPipeline.filter_by_criteria(...) → subset DataFrame
  • MaterialEvaluationPipeline.get_top_n(n) → top N materials
  • run_full_evaluation(...) → (results_df, pipeline)
  • compare_interaction_parameters(materials, {param_sets}) → sensitivity table

PHYSICAL CONSTANTS
──────────────────
  K_B = 8.617e-5  # Boltzmann constant (eV/K)
  F = 96485.3     # Faraday constant (C/mol)
  T_default = 300 # Temperature (K)

TYPICAL PARAMETER RANGES
────────────────────────
  α (transport exponent):     1.5 - 3.0
  β (stability exponent):     1.0 - 2.5
  γ (frustration penalty):    2.0 - 5.0

EXAMPLE WORKFLOW
────────────────
from evaluation_pipeline import run_full_evaluation

results, pipeline = run_full_evaluation(
    "path/to/cifs/",
    model_dir="path/to/models/",
    limit=100
)

# Filter promising candidates
promising = pipeline.filter_by_criteria(
    score_min=75,
    feasibility_min=0.6,
    stability_state_min=0.5
)

# Sensitivity analysis
sensitivity_df = compare_interaction_parameters(
    materials,
    {
        'default': InteractionParameters(2.0, 1.5, 3.0),
        'transport_favoring': InteractionParameters(3.0, 1.0, 2.0),
        'stability_favoring': InteractionParameters(1.5, 2.5, 2.5),
    }
)
""")

print("\n✅ Framework loaded and ready to use!")